In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import numpy as np
import polars as pl
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from py3dbp import Packer, Bin, Item

def create_bin(df,vehicle_number):

    rows = df.filter(pl.col('Vehicle_ID') == vehicle_number).filter(pl.col('Destination') != 'Depot')
    rows = rows.with_columns(
        pl.int_range(rows.height,0,-1).alias('rank')
    )
    packer = Packer()

    truck = Bin(
        partno='Truck',
        WHD=(160, 180, 280),
        max_weight=999999,
        put_type=1        
    )
    packer.addBin(truck)  


    for row in rows.iter_rows(named=True):
        item = Item(
            partno=row["Box_ID"],
            name=row["Box_ID"],
            typeof='cube',
            WHD=(
                int(row["Box_Width"]),
                int(row["Box_Height"]),
                int(row["Box_Length"])
            ),
            weight=1,
            level=row["rank"],      
            updown=True,
            loadbear=999999,
            color='#FFCC00'
        )
        packer.addItem(item)  

    packer.pack(
        fix_point=True,
        check_stable=False,
        bigger_first=False 
    )
    return truck,packer

In [2]:
from collections import defaultdict

packing = defaultdict()
df = pl.read_csv('/Users/jameslee/cj/cj_challenge/route_optimization/routes_df.csv')
unfitted = []
for x in range((df.select(['Vehicle_ID']).max()+1).item()):
    truck,packer = create_bin(df,x)
    unfitted.append(len(truck.unfitted_items))
    packing[x] = packer


In [3]:
if sum(unfitted) == 0:
    print("All items fitted")
else:
    print(f"Unfitted items: {sum(unfitted)}")

All items fitted


In [ ]:
import polars as pl

records = []
for item in packing[0].bins[0].items:
    records.append({
        "Box_ID": item.name,
        "Lower_Left_X": item.position[0],
        "Lower_Left_Y": item.position[2], 
        "Lower_Left_Z": item.position[1],
        "Box_Width": item.width,
        "Box_Length": item.depth,
        "Box_Height": item.height
    })

qwer = pl.DataFrame(records)

qwer = qwer.with_columns([
    pl.col("Lower_Left_X").cast(pl.Float64),
    pl.col("Lower_Left_Y").cast(pl.Float64),
    pl.col("Lower_Left_Z").cast(pl.Float64),
    pl.col("Box_Width").cast(pl.Float64),
    pl.col("Box_Length").cast(pl.Float64),
    pl.col("Box_Height").cast(pl.Float64),
])

In [81]:
colors = plt.get_cmap("tab20", qwer.height)
faces = [
    [0, 1, 2], [0, 2, 3],
    [4, 5, 6], [4, 6, 7],
    [0, 1, 5], [0, 5, 4],
    [1, 2, 6], [1, 6, 5],
    [2, 3, 7], [2, 7, 6],
    [3, 0, 4], [3, 4, 7]
]

fig = go.Figure()

for idx, row in enumerate(qwer.iter_rows(named=True)):
    x, y, z = row["Lower_Left_X"], row["Lower_Left_Y"], row["Lower_Left_Z"]
    dx, dy, dz = row["Box_Width"], row["Box_Length"], row["Box_Height"]
    box_id = row["Box_ID"]

    verts = [
        [x, y, z], [x+dx, y, z], [x+dx, y+dy, z], [x, y+dy, z],
        [x, y, z+dz], [x+dx, y, z+dz], [x+dx, y+dy, z+dz], [x, y+dy, z+dz]
    ]

    x_coords = [v[0] for v in verts]
    y_coords = [v[1] for v in verts]
    z_coords = [v[2] for v in verts]
    i, j, k = zip(*faces)

    rgba = colors(idx)
    hex_color = f'rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})'

    fig.add_trace(go.Mesh3d(
        x=x_coords, y=y_coords, z=z_coords,
        i=i, j=j, k=k,
        color=hex_color,
        opacity=0.6,
        flatshading=True,
        showscale=False
    ))

    fig.add_trace(go.Scatter3d(
        x=[x + dx / 2],
        y=[y + dy / 2],
        z=[z + dz / 2],
        mode='text',
        text=[box_id],
        textposition='middle center',
        textfont=dict(size=10, color='black'),
        showlegend=False
    ))

fig.update_layout(
    scene=dict(
        xaxis=dict(title='X (Width)', autorange='reversed'),
        yaxis=dict(title='Y (Depth)'),
        zaxis=dict(title='Z (Height)'),
        aspectmode='data',

    ),
    title='3D Package View',
    width=1000,
    height=1000
)


In [97]:
df.filter(pl.col('Vehicle_ID') == 0).head()

Vehicle_ID,Route_Order,Destination,Order_Number,Box_ID,Stacking_Order,Lower_Left_X,Lower_Left_Y,Lower_Left_Z,Longitude,Latitude,Box_Width,Box_Length,Box_Height,Volume
i64,i64,str,i64,str,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64
0,1,"""Depot""",null,null,null,null,null,null,null,null,null,null,null,null
0,2,"""D_00044""",61,"""B_00061""",0,0,0,0,129.073941,35.178325,30,50,40,60000
0,3,"""D_00013""",20,"""B_00020""",0,0,0,0,129.068896,35.176189,30,40,30,36000
0,4,"""D_00014""",21,"""B_00021""",0,0,0,0,129.060913,35.161701,30,50,40,60000
0,5,"""D_00014""",22,"""B_00022""",0,0,0,0,129.060913,35.161701,30,50,40,60000


In [4]:
fuck = pl.DataFrame()
for i in range((df.select(['Vehicle_ID']).max()+1).item()):
    records = []
    for item in packing[i].bins[0].items:
        records.append({
        "Box_ID": item.name,
        "Lower_Left_X": item.position[0],
        "Lower_Left_Y": item.position[2], 
        "Lower_Left_Z": item.position[1],
        "Box_Width": item.width,
        "Box_Length": item.depth,
        "Box_Height": item.height
    })

    qwer = pl.DataFrame(records)

    qwer = qwer.with_columns([
    pl.col("Lower_Left_X").cast(pl.Float64),
    pl.col("Lower_Left_Y").cast(pl.Float64),
    pl.col("Lower_Left_Z").cast(pl.Float64),
    pl.col("Box_Width").cast(pl.Float64),
    pl.col("Box_Length").cast(pl.Float64),
    pl.col("Box_Height").cast(pl.Float64),
    ])
    fuck = fuck.vstack(qwer)

In [5]:
joined = df.join(fuck, on='Box_ID', how='left', suffix='_qwer')

cols_to_replace = ['Lower_Left_X', 'Lower_Left_Y', 'Lower_Left_Z']

df = joined.with_columns([
    pl.coalesce([pl.col(f"{col}_qwer"), pl.col(col)]).alias(col)
    for col in cols_to_replace
]).select([col for col in joined.columns if not col.endswith('_qwer')])

In [6]:
df = df.with_columns([
    pl.when(pl.col("Destination") != "Depot")
    .then(
        pl.col("Route_Order")
        .rank("dense", descending=True)
        .over("Vehicle_ID")
        .cast(pl.Int64)
    )
    .otherwise(None)
    .alias("Stacking_Order")
])

In [ ]:
df.write_csv('/Users/jameslee/cj/cj_challenge/load_optimization/deeppack2.csv')